# 4.8 · 非线性回归 / Nonlinear Regression

> **课程定位 / Where this fits**
> **Part 4 第 8 课**。4.3 多项式、4.7 GLM 都还是"对参数线性"。但很多**物理/生物/经济**模型本身参数非线性——如指数衰减 $y = a e^{-bx}$（a,b 相乘缠绕）、米氏方程、逻辑增长。这类要用 `scipy.optimize.curve_fit`（Levenberg-Marquardt）。
> Many physical/biological models are nonlinear in the parameters themselves. These need scipy's curve_fit (Levenberg-Marquardt).

> 💡 **面试相关 / Interview-relevant**
> - "参数非线性 vs 特征非线性区别" ★★★★（多项式 vs 真非线性）
> - "为什么非线性回归没有闭式解" ★★★（迭代优化）
> - "curve_fit 怎么用 + 初值为什么重要" ★★★

---

## 学习目标 / Learning Objectives
1. 区分**对特征非线性**（多项式, 仍线性模型）vs **对参数非线性**（真非线性回归）。
2. 用 `scipy.optimize.curve_fit` 拟合任意自定义函数。
3. 理解为什么非线性回归**需迭代 + 依赖初值 + 可能陷局部最优**。
4. 拿到参数的**置信区间**（从协方差矩阵, 接 2.9 Fisher）。

## 目录 / TOC
1. [对参数非线性 ⭐](#1)
2. [curve_fit 基础: 指数衰减](#2)
3. [初值的重要性 ⚠](#3)
4. [参数置信区间 (2.9 接口)](#4)
5. [经典非线性模型: 逻辑增长](#5)
6. [小结](#6)


<a id="1"></a>
## 1. 对参数非线性 ⭐ / Nonlinear in the Parameters

**关键区分**（4.3 强调过"线性指参数"）：

| 模型 | 关于特征 | 关于参数 | 类型 |
|---|---|---|---|
| $y = w_0 + w_1 x + w_2 x^2$ | 非线性(曲线) | **线性** | 线性回归(多项式) → 闭式解 |
| $y = a\,e^{bx}$ | 非线性 | **非线性**(a,b 相乘) | **真非线性回归** → 迭代 |
| $y = \frac{a x}{b + x}$ (米氏) | 非线性 | 非线性 | 真非线性回归 |

**为什么真非线性没有闭式解**: 对参数求梯度置 0 得到的方程**无法解析求解**（参数相互缠绕）→ 必须**迭代数值优化**（0.10 节)。`curve_fit` 默认用 **Levenberg-Marquardt**（高斯-牛顿 + 梯度下降的自适应混合, 0.10 提过 LM）。
No closed form because setting the gradient to zero gives equations with entangled parameters — must iterate. curve_fit uses Levenberg-Marquardt.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)
print("scipy curve_fit 已就绪")


<a id="2"></a>
## 2. curve_fit 基础: 指数衰减 / Exponential Decay

经典场景: 放射性衰减 / 药物代谢 / 电容放电, $y = a\,e^{-bx} + c$。


In [ ]:
# 真实模型 + 噪声 / true model + noise
def exp_decay(x, a, b, c):
    return a * np.exp(-b * x) + c

x = np.linspace(0, 5, 60)
true_params = [5.0, 0.8, 1.0]      # a, b, c
y = exp_decay(x, *true_params) + rng.normal(0, 0.2, len(x))

# 拟合: curve_fit(函数, x, y, 初值) / fit
popt, pcov = curve_fit(exp_decay, x, y, p0=[1, 1, 1])
print(f"真实参数:  a={true_params[0]}, b={true_params[1]}, c={true_params[2]}")
print(f"拟合参数:  a={popt[0]:.3f}, b={popt[1]:.3f}, c={popt[2]:.3f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x, y, alpha=0.5, s=15, label="数据")
ax.plot(x, exp_decay(x, *popt), "r-", lw=2, label=f"拟合 {popt[0]:.1f}·e^(-{popt[1]:.1f}x)+{popt[2]:.1f}")
ax.legend(); ax.set_title("非线性回归: curve_fit 拟合指数衰减")
plt.tight_layout(); plt.show()
print("\ncurve_fit 三件套: 自定义函数 + 数据 + 初值 p0 → 返回最优参数 popt 和协方差 pcov")


<a id="3"></a>
## 3. 初值的重要性 ⚠ / The Importance of Initial Guess

非线性优化是**非凸**的（4.3/0.10 的偏差-方差是凸的, 这里不是）→ **可能陷入局部最优**。**好的初值至关重要**——这是非线性回归 vs 线性回归（凸, 一步到全局最优）的根本区别。


In [ ]:
# 演示: 坏初值导致拟合失败 / bad initial guess fails
def sine_model(x, a, freq, phase):
    return a * np.sin(freq * x + phase)

x2 = np.linspace(0, 10, 100)
y2 = sine_model(x2, 2.0, 1.5, 0.5) + rng.normal(0, 0.2, 100)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# 好初值 (接近真实频率) / good guess near true frequency
try:
    popt_good, _ = curve_fit(sine_model, x2, y2, p0=[1, 1.4, 0])
    axes[0].plot(x2, sine_model(x2, *popt_good), "g-", lw=2, label=f"freq={popt_good[1]:.2f}")
except: pass
axes[0].scatter(x2, y2, alpha=0.3, s=10); axes[0].legend()
axes[0].set_title("好初值 (freq≈1.4): 收敛到真实 1.5 ✓")

# 坏初值 (频率差太远) / bad guess, wrong frequency basin
try:
    popt_bad, _ = curve_fit(sine_model, x2, y2, p0=[1, 5.0, 0], maxfev=5000)
    axes[1].plot(x2, sine_model(x2, *popt_bad), "r-", lw=2, label=f"freq={popt_bad[1]:.2f}")
except: pass
axes[1].scatter(x2, y2, alpha=0.3, s=10); axes[1].legend()
axes[1].set_title("坏初值 (freq=5): 陷局部最优, 拟合失败 ✗")
plt.tight_layout(); plt.show()
print("正弦频率拟合是出名的多局部最优问题 — 初值离真实频率太远就陷在错误的'波峰盆地'")
print("实战: 用领域知识/网格搜索定初值; 或多个初值取最优 (避免局部陷阱)")


<a id="4"></a>
## 4. 参数置信区间 / Parameter Confidence Intervals

`curve_fit` 返回的 `pcov` 是**参数协方差矩阵**（来自 2.9 节的 Fisher 信息 / Hessian 逆）。对角线开根 = 参数标准误 → 算置信区间（2.5）。


In [ ]:
# 用指数衰减的 pcov 算参数 CI / parameter CIs from pcov
popt, pcov = curve_fit(exp_decay, x, y, p0=[1,1,1])
perr = np.sqrt(np.diag(pcov))        # 参数标准误 = 协方差对角线开根 (2.9)

print(f"{'参数':<6} {'估计':>8} {'标准误':>8} {'95% CI':>22} {'真值':>6}")
for name, est, se, true in zip("abc", popt, perr, true_params):
    lo, hi = est - 1.96*se, est + 1.96*se
    inside = "✓" if lo <= true <= hi else "✗"
    print(f"{name:<6} {est:>8.3f} {se:>8.3f} [{lo:.3f}, {hi:.3f}]{'':>2} {true:>6} {inside}")
print("\npcov 来自 2.9 的 Fisher/Hessian; 对角线开根=SE; ±1.96·SE = 95% CI (2.5)")
print("→ 非线性回归同样能做完整统计推断, 不只是点估计")


<a id="5"></a>
## 5. 经典非线性模型: 逻辑增长 / Logistic Growth

S 形增长（人口、产品采用、疫情累计）: $y = \frac{L}{1 + e^{-k(x - x_0)}}$（L=饱和值, k=增长率, x₀=拐点）。


In [ ]:
def logistic_growth(x, L, k, x0):
    return L / (1 + np.exp(-k * (x - x0)))

# 模拟产品累计采用量 / cumulative adoption
x3 = np.linspace(0, 20, 80)
y3 = logistic_growth(x3, L=1000, k=0.6, x0=10) + rng.normal(0, 25, 80)

popt3, pcov3 = curve_fit(logistic_growth, x3, y3, p0=[800, 0.5, 8])
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x3, y3, alpha=0.4, s=15, label="累计采用量")
ax.plot(x3, logistic_growth(x3, *popt3), "r-", lw=2,
        label=f"拟合: L={popt3[0]:.0f}, k={popt3[1]:.2f}, x0={popt3[2]:.1f}")
ax.axhline(popt3[0], color="g", ls="--", alpha=0.5, label=f"饱和值 L≈{popt3[0]:.0f}")
ax.legend(); ax.set_title("逻辑增长曲线: 预测最终饱和量 (产品/疫情建模)")
plt.tight_layout(); plt.show()
print(f"模型外推: 最终饱和采用量 L = {popt3[0]:.0f} (真实 1000)")
print("逻辑增长是非线性回归的杀手应用: 从早期数据预测最终天花板 (新冠/新产品建模常用)")


<a id="6"></a>
## 6. 小结 / Summary

```
对参数非线性 (a·e^{bx}, 米氏, 逻辑增长) → 真非线性回归
  vs 对特征非线性但对参数线性 (多项式) → 仍是线性回归(闭式)
无闭式解 (参数缠绕) → 迭代优化, curve_fit 用 Levenberg-Marquardt
非凸 → 依赖初值 p0, 可能陷局部最优 (正弦频率经典坑) ⚠
pcov 给参数协方差 (2.9 Fisher) → SE → CI (2.5), 完整统计推断
杀手应用: 逻辑增长从早期数据预测饱和天花板
```

### 💡 面试速查
1. **多项式≠非线性回归**: 前者对参数线性(闭式), 后者对参数非线性(迭代)
2. **非凸 → 初值关键**: 坏初值陷局部最优
3. **curve_fit(f, x, y, p0)**: 函数+数据+初值
4. **pcov 对角线开根 = 参数 SE** (2.9 Fisher)
5. **逻辑增长**预测饱和天花板

### 下一节
**4.9 支持向量回归 SVR**——前面的非线性靠人工指定函数形式。SVR 用**核技巧**自动处理非线性, 而且只关心"管道外"的点 (ε-不敏感损失)。
